In [ ]:
features = [
    'payment_value',
    'price',
    'freight_value',
    'delivery_time_days',
    'delivery_delay_days',
    'review_text_length'
]

df_model = df.dropna(subset=features + ['target'])

X = df_model[features]
y = df_model['target']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5
)

xgb.fit(X_train, y_train)

In [ ]:
results = []

for name, model in [
    ("LogReg", model),
    ("RF", rf),
    ("GB", gb),
    ("XGB", xgb)
]:
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba)
    })

import pandas as pd
pd.DataFrame(results)

In [2]:
import pandas as pd

df = pd.read_csv('../data/processed/df.csv')
# 03_experiments.ipynb
# ML Experiments Notebook

# =========================
# 1. Imports
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Optional (if installed)
try:
    from xgboost import XGBClassifier
    xgb_available = True
except:
    xgb_available = False

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# =========================
# 2. Load data
# =========================
# If you saved processed dataset, use it:
# df = pd.read_csv('../data/processed/df.csv')

# Otherwise assume df already exists from EDA notebook

# =========================
# 3. Features
# =========================
features = [
    'payment_value',
    'price',
    'freight_value',
    'delivery_time_days',
    'delivery_delay_days',
    'review_text_length'
]

# safety check
required_cols = features + ['target']
df_model = df.dropna(subset=required_cols)

X = df_model[features]
y = df_model['target']

# =========================
# 4. Train / test split
# =========================
df_model = df_model.sort_values('order_purchase_timestamp')

train_size = int(len(df_model) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]
y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

# =========================
# 5. Models
# =========================
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

if xgb_available:
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss'
    )

# =========================
# 6. Training + evaluation
# =========================
results = []

for name, model in models.items():
    print(f"Training {name}...")
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    # probability (some models may not have predict_proba)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc = roc_auc_score(y_test, y_proba)
    else:
        roc = None
    
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc
    })

# =========================
# 7. Results table
# =========================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="f1", ascending=False)

results_df

# =========================
# 8. Best model
# =========================
best_model_name = results_df.iloc[0]["model"]
print("Best model:", best_model_name)

best_model = models[best_model_name]

# Feature importance (if available)
if hasattr(best_model, "feature_importances_"):
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    importance = pd.DataFrame({
        "feature": features,
        "importance": best_model.feature_importances_
    }).sort_values(by="importance", ascending=False)
    
    sns.barplot(data=importance, x="importance", y="feature")
    plt.title("Feature Importance")
    plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/df.csv'